# Haiku for three NLP tasks

Using **Claude Haiku 4.5** to:

1. **Sentiment analysis** — 3 examples
2. **Medical note summarization** — 2 examples
3. **Translation** — one medical note, English → Spanish

> The clinical notes below are **synthetic** (invented for teaching, no real patient data). For real clinical workloads you need a signed BAA with Anthropic and appropriate PHI handling — don't paste real patient data into an unapproved account.

## Setup

Set `ANTHROPIC_API_KEY` in your shell before launching Jupyter; the cell below is a fallback.

In [1]:
%pip install anthropic

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from anthropic import Anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your Anthropic API key: ")

client = Anthropic()
MODEL = "claude-haiku-4-5-20251001"

def ask(prompt, system=None, max_tokens=500):
    """Send one prompt to Haiku and return the text reply."""
    kwargs = {"model": MODEL, "max_tokens": max_tokens,
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    return client.messages.create(**kwargs).content[0].text

## 1. Sentiment analysis (3 examples)

A **system prompt** pins the model to a fixed, machine-readable format — the key trick for classification tasks. Here we ask for the label plus a confidence score.

In [3]:
SENTIMENT_SYSTEM = (
    "You are a sentiment classifier. Classify the user's text as "
    "POSITIVE, NEGATIVE, or NEUTRAL. "
    "Reply on a single line in exactly this format: LABEL (confidence 0-100%). "
    "Do not explain."
)

reviews = [
    "Absolutely loved it — fast shipping and the quality exceeded my expectations!",
    "The app crashes every time I try to log in. Completely useless.",
    "It arrived on the scheduled date. Packaging was standard.",
]

for text in reviews:
    result = ask(text, system=SENTIMENT_SYSTEM, max_tokens=20)
    print(f"{result:<28} | {text}")

POSITIVE (95%)               | Absolutely loved it — fast shipping and the quality exceeded my expectations!
NEGATIVE (98%)               | The app crashes every time I try to log in. Completely useless.
NEUTRAL (85%)                | It arrived on the scheduled date. Packaging was standard.


## 2. Medical note summarization (2 examples)

Summarize a longer clinical note into a short, structured summary. The system prompt sets the role and the output shape.

In [4]:
SUMMARY_SYSTEM = (
    "You are a clinical documentation assistant. Summarize the note in 3 bullet points: "
    "(1) chief complaint / diagnosis, (2) key findings, (3) plan. "
    "Be concise and use only information present in the note."
)

notes = [
    """Patient is a 58-year-old male presenting with a 3-day history of productive cough,
fever up to 38.9C, and shortness of breath. History of type 2 diabetes and hypertension.
On exam, decreased breath sounds and crackles at the right lung base. SpO2 93% on room air.
Chest X-ray shows right lower lobe consolidation. WBC 14.2. Diagnosed with community-acquired
pneumonia. Started on oral amoxicillin-clavulanate. Advised rest, fluids, and follow-up in
48-72 hours or sooner if symptoms worsen.""",

    """34-year-old female with no significant past medical history presents with intermittent
palpitations and lightheadedness over the past 2 weeks, worse with caffeine. Denies chest pain
or syncope. Vitals stable, BP 118/74, HR 88 regular. ECG shows normal sinus rhythm with occasional
premature atrial contractions. TSH and electrolytes within normal limits. Reassured patient;
advised reducing caffeine intake. Ordered 48-hour Holter monitor to characterize palpitations.""",
]

for i, note in enumerate(notes, 1):
    print(f"--- Note {i} summary ---")
    print(ask(note, system=SUMMARY_SYSTEM, max_tokens=250))
    print()

--- Note 1 summary ---
# Clinical Documentation Summary

1. **Chief Complaint/Diagnosis:** 58-year-old male with community-acquired pneumonia presenting with 3-day history of productive cough, fever (38.9°C), and shortness of breath.

2. **Key Findings:** Decreased breath sounds and crackles at right lung base; SpO2 93% on room air; chest X-ray shows right lower lobe consolidation; elevated WBC 14.2.

3. **Plan:** Initiated oral amoxicillin-clavulanate; advised rest and increased fluid intake; follow-up in 48-72 hours or sooner if symptoms worsen.

--- Note 2 summary ---
# Clinical Documentation Summary

• **Chief Complaint/Diagnosis:** 34-year-old female with intermittent palpitations and lightheadedness over 2 weeks, triggered by caffeine; ECG shows occasional premature atrial contractions (PACs).

• **Key Findings:** Vital signs stable (BP 118/74, HR 88 regular), normal sinus rhythm on ECG, TSH and electrolytes within normal limits; no chest pain or syncope reported.

• **Plan:** Pa

## 3. Translation: English → Spanish

Translate one clinical note, preserving medical terminology.

In [5]:
TRANSLATE_SYSTEM = (
    "You are a medical translator. Translate the clinical note from English into Spanish. "
    "Preserve medical terminology and units accurately. Output only the translation."
)

note_to_translate = notes[0]

print("ENGLISH (original):\n")
print(note_to_translate)
print("\n" + "=" * 60 + "\n")
print("SPANISH (translation):\n")
print(ask(note_to_translate, system=TRANSLATE_SYSTEM, max_tokens=400))

ENGLISH (original):

Patient is a 58-year-old male presenting with a 3-day history of productive cough,
fever up to 38.9C, and shortness of breath. History of type 2 diabetes and hypertension.
On exam, decreased breath sounds and crackles at the right lung base. SpO2 93% on room air.
Chest X-ray shows right lower lobe consolidation. WBC 14.2. Diagnosed with community-acquired
pneumonia. Started on oral amoxicillin-clavulanate. Advised rest, fluids, and follow-up in
48-72 hours or sooner if symptoms worsen.


SPANISH (translation):

El paciente es un varón de 58 años que se presenta con un historial de 3 días de tos productiva, fiebre hasta 38.9°C y disnea. Antecedentes de diabetes mellitus tipo 2 e hipertensión.

En el examen físico, disminución de los sonidos respiratorios y crepitantes en la base del pulmón derecho. SpO2 93% en aire ambiente.

La radiografía de tórax muestra consolidación del lóbulo inferior derecho. GB 14.2. Se diagnostica neumonía adquirida en la comunidad. Se inic